# 11 · Fractional derivatives

Fractional calculus generalises `dⁿ/dxⁿ` to non-integer order α. Unlike the rest
of omnibias these operators are **non-local** — the value at a point depends on
the whole history/domain — so they are honest *grid-based numerical
approximations*, **not** closed-form σ-tower derivatives. `omnibias-fractional`
ships Grünwald–Letnikov / Riemann–Liouville / Caputo (uniform grid) and a
spectral (FFT) variant for periodic domains.

Reference identity: `Dᵅ xᵖ = Γ(p+1)/Γ(p+1−α) · x^{p−α}`.

In [ ]:
import math
import sys

import numpy as np
import torch
import matplotlib.pyplot as plt

sys.path.insert(0, ".")
from _style import set_style, PRIMARY, ACCENT, WARM
set_style()

torch.set_default_dtype(torch.float64)

from omnibias.fractional.torch.ops.fractional import (
    grunwald_letnikov,
    spectral_fractional,
)

print("ready")

## Grünwald–Letnikov derivative of `xᵖ`

Take `f(x) = x²` on `[0, 2]` and order `α = 0.5`. The GL estimate converges to
the analytic `Dᵅ xᵖ` as the grid is refined (the largest error sits near the
left boundary, where the non-local memory is truncated, so we measure the
interior error).

In [ ]:
p, alpha, X = 2.0, 0.5, 2.0
coef = math.gamma(p + 1) / math.gamma(p + 1 - alpha)


def gl_on_grid(N):
    x = torch.linspace(0.0, X, N)
    h = X / (N - 1)
    d = grunwald_letnikov(x**p, alpha=alpha, h=h)
    exact = coef * x ** (p - alpha)
    lo = N // 5  # skip the left-boundary memory-truncation region
    return x.numpy(), d.numpy(), exact.numpy(), float((d[lo:] - exact[lo:]).abs().max())


Ns = [50, 100, 200, 400, 800]
errs = [gl_on_grid(N)[3] for N in Ns]
x, d, exact, _ = gl_on_grid(400)
for N, e in zip(Ns, errs):
    print(f"N={N:4d}   interior max error = {e:.3e}")

## Spectral fractional derivative (periodic)

On a periodic domain the FFT multiplier `(ik)ᵅ` gives a spectrally-accurate
fractional derivative. Two checks: order `α = 1` recovers the ordinary
derivative of `sin`, and the **semigroup** property `Dᵅ Dᵅ = D^{2α}` holds for
band-limited inputs.

In [ ]:
N = 256
L = 2.0 * np.pi
xs = torch.linspace(0.0, L, N + 1)[:-1]
f = torch.sin(xs)

d1 = spectral_fractional(f, alpha=1.0, length=L)
print("order-1 vs cos(x):  max error =", float((d1.real - torch.cos(xs)).abs().max()))

once = spectral_fractional(f, alpha=1.0, length=L)
twice_half = spectral_fractional(spectral_fractional(f, alpha=0.5, length=L),
                                 alpha=0.5, length=L)
print("D^0.5 D^0.5 vs D^1: max error =", float((twice_half - once).abs().max()))

In [ ]:
fig, (axl, axr) = plt.subplots(1, 2, figsize=(11, 4.2))

axl.loglog(Ns, errs, "o-", color=ACCENT, label="interior max error")
axl.loglog(Ns, [errs[0] * (Ns[0] / n) for n in Ns], "--", color=WARM, label="O(1/N)")
axl.set_xlabel("grid points N")
axl.set_ylabel("error")
axl.set_title(r"GL $D^{0.5} x^2$ converges")
axl.legend()

axr.plot(x, exact, color=PRIMARY, lw=2.5, label=r"analytic $D^{0.5}x^2$")
axr.plot(x, d, "--", color=ACCENT, label="GL (N=400)")
axr.set_xlabel("x")
axr.set_title("Grünwald–Letnikov vs analytic")
axr.legend()

plt.tight_layout()
plt.show()

## Takeaway

`omnibias-fractional` adds fractional-PDE operators with an **honest** label:
grid / spectral approximations with a documented error budget, not exact σ-tower
derivatives. Grid refinement and spectral accuracy behave exactly as the theory
predicts.

Next: **[12 · Score & Fokker–Planck](12_score_ou_generator.ipynb)**.